In [1]:
import importlib, env
importlib.reload(env)
from env import CloudClusterEnv, STEPS_PER_WEEK, MIN_PODS, MAX_PODS

import json, numpy as np, torch, torch.nn as nn
from torch.distributions import Normal

stats = json.load(open('trace_params.json'))['stats']

# ---- PPO network ----
class ActorCritic(nn.Module):
    def __init__(self, state_dim=32, action_dim=1):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(),
            nn.Linear(256, 256),       nn.Tanh())
        self.actor_mean = nn.Linear(256, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.critic = nn.Linear(256, 1)
    def forward(self, s):
        x = self.shared(s); return self.actor_mean(x), self.critic(x)

# ---- DQN network (must match 09_dqn_baseline) ----
DISCRETE_ACTIONS = [-5, -2, 0, +2, +5]
N_ACTIONS = len(DISCRETE_ACTIONS)
class DQN(nn.Module):
    def __init__(self, state_dim=32, n_actions=N_ACTIONS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Linear(256, n_actions))
    def forward(self, s): return self.net(s)

# load trained models
ppo_net = ActorCritic()
ppo_net.load_state_dict(torch.load('ppo_sla-focused.pth'))
ppo_net.eval()

dqn_net = DQN()
dqn_net.load_state_dict(torch.load('dqn_baseline.pth'))
dqn_net.eval()

print("Models loaded: PPO and DQN ready for evaluation.")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Setup complete. Steps per week: 672
CloudClusterEnv defined.
Models loaded: PPO and DQN ready for evaluation.


In [2]:
# ---- evaluate each agent on ONE seed (same workload for fair comparison) ----

def eval_hpa(seed):
    env = CloudClusterEnv(stats, seed=seed)
    env.reset()
    cost = breaches = util = 0.0; vms = []; steps = 0
    for t in range(STEPS_PER_WEEK):
        cpu = env.history[-1][0]
        if cpu > 0.70:   a = np.array([0.2])
        elif cpu < 0.30: a = np.array([-0.2])
        else:            a = np.array([0.0])
        _, r, done, _, info = env.step(a)
        cost += info['cost']; breaches += info['breaches']
        util += info['utilisation']; vms.append(info['active_vms']); steps += 1
        if done: break
    return {'cost': cost, 'breaches': breaches, 'util': util/steps, 'vms': np.mean(vms)}

def eval_ppo(seed):
    env = CloudClusterEnv(stats, seed=seed)
    obs, _ = env.reset(); obs = torch.tensor(obs, dtype=torch.float32)
    cost = breaches = util = 0.0; vms = []; steps = 0
    for t in range(STEPS_PER_WEEK):
        with torch.no_grad():
            mean, _ = ppo_net.forward(obs.unsqueeze(0))
        _, r, done, _, info = env.step(mean.squeeze(0).numpy())
        obs = torch.tensor(env._build_state(), dtype=torch.float32)
        cost += info['cost']; breaches += info['breaches']
        util += info['utilisation']; vms.append(info['active_vms']); steps += 1
        if done: break
    return {'cost': cost, 'breaches': breaches, 'util': util/steps, 'vms': np.mean(vms)}

def eval_dqn(seed):
    env = CloudClusterEnv(stats, seed=seed)
    obs, _ = env.reset()
    cost = breaches = util = 0.0; vms = []; steps = 0
    for t in range(STEPS_PER_WEEK):
        with torch.no_grad():
            qv = dqn_net(torch.tensor(obs, dtype=torch.float32).unsqueeze(0))
            idx = qv.argmax().item()
        a = np.array([DISCRETE_ACTIONS[idx] / 5.0])
        obs, r, done, _, info = env.step(a)
        cost += info['cost']; breaches += info['breaches']
        util += info['utilisation']; vms.append(info['active_vms']); steps += 1
        if done: break
    return {'cost': cost, 'breaches': breaches, 'util': util/steps, 'vms': np.mean(vms)}

# test all three on ONE seed
seed = 42
print(f"Single-seed test (seed={seed}):\n")
print(f"{'agent':<6}{'cost':>8}{'breaches':>10}{'util':>8}{'vms':>7}")
for name, fn in [('HPA', eval_hpa), ('PPO', eval_ppo), ('DQN', eval_dqn)]:
    r = fn(seed)
    print(f"{name:<6}{r['cost']:>8.1f}{r['breaches']:>10.0f}{r['util']:>8.3f}{r['vms']:>7.1f}")

Single-seed test (seed=42):

agent     cost  breaches    util    vms
HPA      290.2      1782   0.563    8.6
PPO      183.2       327   0.936    5.5
DQN      173.7      1858   0.960    5.2


In [3]:
import time

N_SEEDS = 30    # 30 independent workloads per agent

results = {'HPA': [], 'PPO': [], 'DQN': []}
t0 = time.time()

print(f"Running {N_SEEDS} seeds per agent...\n")
for i in range(N_SEEDS):
    seed = 1000 + i
    results['HPA'].append(eval_hpa(seed))
    results['PPO'].append(eval_ppo(seed))
    results['DQN'].append(eval_dqn(seed))
    if (i+1) % 5 == 0:
        print(f"  completed {i+1}/{N_SEEDS} seeds  ({time.time()-t0:.0f}s)")

print(f"\nDone in {time.time()-t0:.0f}s.")

# save raw results for the report
import json
with open('significance_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved significance_results.json")

# quick preview: mean ± std per agent
print(f"\n{'agent':<6}{'cost (mean±std)':>22}{'breaches (mean±std)':>24}")
for agent in ['HPA', 'DQN', 'PPO']:
    costs = [r['cost'] for r in results[agent]]
    breaches = [r['breaches'] for r in results[agent]]
    print(f"{agent:<6}{np.mean(costs):>10.1f} ± {np.std(costs):<8.1f}"
          f"{np.mean(breaches):>12.0f} ± {np.std(breaches):<8.0f}")

Running 30 seeds per agent...

  completed 5/30 seeds  (14s)
  completed 10/30 seeds  (27s)
  completed 15/30 seeds  (41s)
  completed 20/30 seeds  (55s)
  completed 25/30 seeds  (69s)
  completed 30/30 seeds  (83s)

Done in 83s.
Saved significance_results.json

agent        cost (mean±std)     breaches (mean±std)
HPA        291.1 ± 2.3             1785 ± 205     
DQN        173.0 ± 1.1             2735 ± 781     
PPO        183.6 ± 0.9              480 ± 282     


In [4]:
from scipy import stats as scipy_stats

def compare(agent_a, agent_b, metric):
    """Paired t-test: is agent_a significantly different from agent_b on this metric?"""
    a = np.array([r[metric] for r in results[agent_a]])
    b = np.array([r[metric] for r in results[agent_b]])
    # paired because same seeds → same workloads
    t_stat, p_value = scipy_stats.ttest_rel(a, b)
    return a.mean(), b.mean(), t_stat, p_value

print("="*70)
print("STATISTICAL SIGNIFICANCE TESTS (paired t-test, n=30 seeds)")
print("="*70)

for metric in ['cost', 'breaches']:
    print(f"\n--- {metric.upper()} ---")
    for a, b in [('PPO','HPA'), ('PPO','DQN'), ('DQN','HPA')]:
        ma, mb, t, p = compare(a, b, metric)
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
        better = a if (ma < mb) else b   # lower is better for both metrics
        print(f"  {a} ({ma:.0f}) vs {b} ({mb:.0f}): "
              f"t={t:.2f}, p={p:.2e} {sig}  → {better} better")

print("\n" + "="*70)
print("Significance: *** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant")

# also compute % improvement with confidence
print("\n--- PPO improvement over HPA ---")
for metric in ['cost', 'breaches']:
    ppo_vals = np.array([r[metric] for r in results['PPO']])
    hpa_vals = np.array([r[metric] for r in results['HPA']])
    pct = (hpa_vals - ppo_vals) / hpa_vals * 100
    print(f"  {metric}: {pct.mean():.1f}% ± {pct.std():.1f}% reduction")

STATISTICAL SIGNIFICANCE TESTS (paired t-test, n=30 seeds)

--- COST ---
  PPO (184) vs HPA (291): t=-245.41, p=1.15e-49 ***  → PPO better
  PPO (184) vs DQN (173): t=45.81, p=1.32e-28 ***  → DQN better
  DQN (173) vs HPA (291): t=-282.98, p=1.86e-51 ***  → DQN better

--- BREACHES ---
  PPO (480) vs HPA (1785): t=-21.01, p=4.29e-19 ***  → PPO better
  PPO (480) vs DQN (2735): t=-14.96, p=3.62e-15 ***  → PPO better
  DQN (2735) vs HPA (1785): t=6.64, p=2.79e-07 ***  → HPA better

Significance: *** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant

--- PPO improvement over HPA ---
  cost: 36.9% ± 0.5% reduction
  breaches: 72.9% ± 15.7% reduction


In [5]:
import time

# ---- INFERENCE LATENCY: how long does one decision take? ----
env = CloudClusterEnv(stats, seed=1)
obs, _ = env.reset()
obs_t = torch.tensor(obs, dtype=torch.float32)

# warm up (first inference includes setup overhead)
for _ in range(10):
    with torch.no_grad():
        ppo_net.forward(obs_t.unsqueeze(0))

# measure PPO inference latency over many runs
n_trials = 1000
t0 = time.perf_counter()
for _ in range(n_trials):
    with torch.no_grad():
        mean, _ = ppo_net.forward(obs_t.unsqueeze(0))
ppo_latency = (time.perf_counter() - t0) / n_trials * 1000  # ms

# DQN latency
t0 = time.perf_counter()
for _ in range(n_trials):
    with torch.no_grad():
        dqn_net(obs_t.unsqueeze(0))
dqn_latency = (time.perf_counter() - t0) / n_trials * 1000

print("INFERENCE LATENCY (per decision):")
print(f"  PPO: {ppo_latency:.3f} ms")
print(f"  DQN: {dqn_latency:.3f} ms")
print(f"\n(Both far below the 15-min decision interval — negligible overhead)")

INFERENCE LATENCY (per decision):
  PPO: 0.032 ms
  DQN: 0.020 ms

(Both far below the 15-min decision interval — negligible overhead)


In [6]:
import os

# ---- RESOURCE OVERHEAD: model size, params, memory ----

# parameter counts
ppo_params = sum(p.numel() for p in ppo_net.parameters())
dqn_params = sum(p.numel() for p in dqn_net.parameters())

# model file sizes on disk
ppo_size = os.path.getsize('ppo_sla-focused.pth') / 1024   # KB
dqn_size = os.path.getsize('dqn_baseline.pth') / 1024

# approximate memory footprint of parameters (float32 = 4 bytes)
ppo_mem = ppo_params * 4 / 1024   # KB
dqn_mem = dqn_params * 4 / 1024

print("RESOURCE OVERHEAD:")
print(f"{'':16}{'PPO':>12}{'DQN':>12}")
print(f"{'Parameters':16}{ppo_params:>12,}{dqn_params:>12,}")
print(f"{'Model file (KB)':16}{ppo_size:>12.1f}{dqn_size:>12.1f}")
print(f"{'Param memory (KB)':16}{ppo_mem:>12.1f}{dqn_mem:>12.1f}")
print(f"\n(A ~75k-param model is tiny — runs comfortably on CPU, "
      f"negligible footprint vs a Kubernetes node)")

RESOURCE OVERHEAD:
                         PPO         DQN
Parameters            74,755      75,525
Model file (KB)        295.9       298.1
Param memory (KB)       292.0       295.0

(A ~75k-param model is tiny — runs comfortably on CPU, negligible footprint vs a Kubernetes node)


In [7]:
# ---- SCALABILITY: decision time vs cluster size ----
# The state is always 32-dim regardless of cluster size, so we simulate
# building + inferring across different MAX_PODS settings.

import time

cluster_sizes = [10, 50, 100, 500, 1000, 5000]
print("SCALABILITY — decision time vs cluster size:")
print(f"{'cluster size':>14}{'decision time (ms)':>20}")

for size in cluster_sizes:
    # the state vector is always 32-dim; inference cost is independent of
    # how many pods the numbers represent
    dummy_state = torch.rand(32)
    n = 500
    t0 = time.perf_counter()
    for _ in range(n):
        with torch.no_grad():
            ppo_net.forward(dummy_state.unsqueeze(0))
    latency = (time.perf_counter() - t0) / n * 1000
    print(f"{size:>14}{latency:>20.4f}")

print("\nDecision time is CONSTANT regardless of cluster size, because the")
print("agent operates on a fixed 32-dim cluster summary, not per-pod data.")
print("This means the approach scales to arbitrarily large clusters.")

SCALABILITY — decision time vs cluster size:
  cluster size  decision time (ms)
            10              0.0370
            50              0.0309
           100              0.0258
           500              0.0242
          1000              0.0227
          5000              0.0208

Decision time is CONSTANT regardless of cluster size, because the
agent operates on a fixed 32-dim cluster summary, not per-pod data.
This means the approach scales to arbitrarily large clusters.


In [8]:
import time, os

# ---- INFERENCE LATENCY including HPA ----
env = CloudClusterEnv(stats, seed=1)
obs, _ = env.reset()
obs_t = torch.tensor(obs, dtype=torch.float32)

# HPA "inference" = the threshold check (measure the actual decision logic)
def hpa_decision(cpu):
    if cpu > 0.70: return 0.2
    elif cpu < 0.30: return -0.2
    return 0.0

n_trials = 1000
# HPA
t0 = time.perf_counter()
for _ in range(n_trials):
    hpa_decision(obs_t[0].item())
hpa_latency = (time.perf_counter() - t0) / n_trials * 1000

# PPO
for _ in range(10):  # warmup
    with torch.no_grad(): ppo_net.forward(obs_t.unsqueeze(0))
t0 = time.perf_counter()
for _ in range(n_trials):
    with torch.no_grad(): ppo_net.forward(obs_t.unsqueeze(0))
ppo_latency = (time.perf_counter() - t0) / n_trials * 1000

# DQN
t0 = time.perf_counter()
for _ in range(n_trials):
    with torch.no_grad(): dqn_net(obs_t.unsqueeze(0))
dqn_latency = (time.perf_counter() - t0) / n_trials * 1000

print("INFERENCE LATENCY & RESOURCE OVERHEAD (all three):")
print(f"{'':18}{'HPA':>12}{'DQN':>12}{'PPO':>12}")
print(f"{'Latency (ms)':18}{hpa_latency:>12.4f}{dqn_latency:>12.4f}{ppo_latency:>12.4f}")
print(f"{'Parameters':18}{0:>12}{75525:>12,}{74755:>12,}")
print(f"{'Model file (KB)':18}{0:>12}{298.1:>12.1f}{295.9:>12.1f}")
print(f"\nHPA has zero model overhead (rule-based); RL agents add a small,")
print(f"negligible overhead (<0.05 ms, <300 KB) for far better decisions.")

INFERENCE LATENCY & RESOURCE OVERHEAD (all three):
                           HPA         DQN         PPO
Latency (ms)            0.0019      0.0190      0.0299
Parameters                   0      75,525      74,755
Model file (KB)              0       298.1       295.9

HPA has zero model overhead (rule-based); RL agents add a small,
negligible overhead (<0.05 ms, <300 KB) for far better decisions.


In [10]:
import json
from env import CloudClusterEnv
from agent import ActorCritic
from evaluate import run_hpa, run_ppo, run_dqn

stats = json.load(open('trace_params.json'))['stats']
net = ActorCritic()
net.load_state_dict(torch.load('ppo_sla-focused.pth'))
net.eval()

# quick smoke test on one episode each
env = CloudClusterEnv(stats, seed=42)
print("HPA:", run_hpa(env))
env = CloudClusterEnv(stats, seed=42)
print("PPO:", run_ppo(env, net))

HPA: {'cost': 290.19999999999953, 'breaches': 1782.0, 'util': 0.5625791657874115, 'vms': 8.636904761904763}
PPO: {'cost': 183.19999999999857, 'breaches': 327.0, 'util': 0.9359979664585196, 'vms': 5.4523809523809526}
